# Santa Barbara County Case Study

This notebook creates detailed maps and analysis for Santa Barbara County water sites.
It follows the same structure as the main CA mapping notebook but focuses on county-level analysis.


## Workflow

1. Load the latest artifacts from the analysis notebook
2. Process coordinates and filter water types (same as main notebook)
3. Filter to Santa Barbara County specifically
4. Create detailed county-level maps and analysis


In [ ]:
# Load artifacts and resolve summary + everything tables
import os
import pandas as pd
from artifacts_io import load_artifacts
import seaborn as sns

tables, meta, run_dir = load_artifacts(out_dir="artifacts")
run_id = os.path.basename(run_dir)
print("Loaded run:", run_dir)
print("Meta:", meta)

# Resolve keys from meta if present; otherwise infer
summary_keys = meta.get("summary_keys", [])
everything_key = meta.get("everything_key")

if not summary_keys:
    # Heuristic: names containing common summary tokens
    name_hits = ("summary", "sum_", "_sum", "agg", "group", "pivot", "tbl", "by_", "_by")
    summary_keys = [k for k in tables if any(tok in k.lower() for tok in name_hits)]

if everything_key is None:
    # Largest table by rows as the 'everything' table
    everything_key = max(tables, key=lambda k: tables[k].shape[0])

print("\nResolved table keys:")
print("  summary_keys :", summary_keys if summary_keys else "(none)")
print("  everything_key:", everything_key)

# Preview heads
for k in tables:
    print(f"\nTable: {k}  shape={tables[k].shape}")
    display(tables[k].head(3))  # Smaller preview for case study

## Coordinate preparation

In [ ]:
# ==== Build combined coordinates from df_summary (same as main notebook) ====
import re
import numpy as np
import pandas as pd

# Select the df_summary table from artifacts
if "tables" not in globals():
    raise RuntimeError("Run the artifacts loading cell first.")

if "df_summary" in tables:
    df_summary = tables["df_summary"].copy()
else:
    # case-insensitive match if key differs in case
    match_keys = [k for k in tables if k.lower() == "df_summary"]
    if not match_keys:
        raise KeyError("I can't find a table named 'df_summary' in artifacts.")
    df_summary = tables[match_keys[0]].copy()

print("df_summary shape:", df_summary.shape)

# Coordinate processing functions (same as main notebook)
def _coerce_num(s):
    return pd.to_numeric(s, errors="coerce")

def _first_nonnull(series_list, index):
    if not series_list:
        return pd.Series(index=index, dtype="float64")
    stacked = pd.concat([_coerce_num(s) for s in series_list], axis=1)
    return stacked.bfill(axis=1).iloc[:, 0]

def _find_cols(df, lat_or_lon, which):
    pat_lat = r'(?:\blat\b|latitude)'
    pat_lon = r'(?:\blon\b|\blong\b|longitude)'
    pat_kind = r'actual' if which == "actual" else r'infer|inferred|estimat|approx'
    cols = []
    for c in df.columns:
        cl = c.lower()
        if (re.search(pat_lat, cl) if lat_or_lon=='lat' else re.search(pat_lon, cl)) and re.search(pat_kind, cl):
            cols.append(c)
    return cols

def _fallback_cols(df, lat_or_lon):
    pat = r'(?:\blat\b|latitude)' if lat_or_lon=='lat' else r'(?:\blon\b|\blong\b|longitude)'
    return [c for c in df.columns if re.search(pat, c.lower())]

# Find coordinate columns
lat_actual_cols = _find_cols(df_summary, 'lat','actual')
lat_infer_cols  = _find_cols(df_summary, 'lat','inferred')
lon_actual_cols = _find_cols(df_summary, 'lon','actual')
lon_infer_cols  = _find_cols(df_summary, 'lon','inferred')

# Fallbacks
if not (lat_actual_cols or lat_infer_cols):
    lat_infer_cols = _fallback_cols(df_summary, 'lat')
if not (lon_actual_cols or lon_infer_cols):
    lon_infer_cols = _fallback_cols(df_summary, 'lon')

lat_names_order = lat_actual_cols + lat_infer_cols
lon_names_order = lon_actual_cols + lon_infer_cols

lat_sources = [df_summary[c] for c in lat_names_order if c in df_summary.columns]
lon_sources = [df_summary[c] for c in lon_names_order if c in df_summary.columns]

if not lat_sources or not lon_sources:
    raise ValueError("Couldn't find latitude/longitude columns.")

# Build combined columns
df_summary["lat_combined"] = _first_nonnull(lat_sources, df_summary.index)
df_summary["lon_combined"] = _first_nonnull(lon_sources, df_summary.index)

# Normalize longitudes and fix lat/lon swaps
def _normalize_lon(x):
    return ((x + 180) % 360) - 180

if (df_summary["lon_combined"] > 180).mean() > 0.2:
    df_summary["lon_combined"] = _normalize_lon(df_summary["lon_combined"])

swap_mask = (~df_summary["lat_combined"].between(-90, 90)) & (df_summary["lon_combined"].between(-90, 90))
if swap_mask.any():
    df_summary.loc[swap_mask, ["lat_combined","lon_combined"]] = (
        df_summary.loc[swap_mask, ["lon_combined","lat_combined"]].to_numpy()
    )

# Record coordinate sources
def _first_used_name(series_list, names):
    out = np.full(len(df_summary), np.nan, dtype=object)
    for s, n in zip(series_list, names):
        take = pd.isna(out) & ~_coerce_num(s).isna()
        out[take.values] = n
    return pd.Series(out, index=df_summary.index)

df_summary["lat_source_col"] = _first_used_name([df_summary[c] for c in lat_names_order], lat_names_order) if lat_names_order else np.nan
df_summary["lon_source_col"] = _first_used_name([df_summary[c] for c in lon_names_order], lon_names_order) if lon_names_order else np.nan

def _safe_tag(val):
    if not isinstance(val, str):
        return "unspecified"
    v = val.lower()
    if "actual" in v:
        return "actual"
    if "infer" in v or "estimat" in v or "approx" in v:
        return "inferred"
    return "unspecified"

df_summary["coord_source"] = [
    _safe_tag(la) if _safe_tag(la) == _safe_tag(lo) else f"{_safe_tag(la)}/{_safe_tag(lo)}"
    for la, lo in zip(df_summary["lat_source_col"], df_summary["lon_source_col"])
]

# Drop invalid coordinates
valid = df_summary["lat_combined"].between(-90, 90) & df_summary["lon_combined"].between(-180, 180)
dropped = (~valid).sum()
df_summary = df_summary.loc[valid].copy()

print(f"Combined coordinates built. Dropped {dropped} rows with invalid/missing coords.")
print("coord_source breakdown:")
print(df_summary["coord_source"].value_counts(dropna=False))

## Water type filtering and color setup

In [ ]:
# ==== WATER TYPE FILTERING AND CONSISTENT COLOR SETUP ====
import geopandas as gpd

print(f"=== WATER TYPE FILTERING ===")

# Filter out unwanted water categories
if 'water_type_clean' in df_summary.columns:
    print("Current water_type_clean distribution:")
    water_counts = df_summary['water_type_clean'].value_counts(dropna=False)
    print(water_counts)
    
    exclude_categories = ['Not water related', 'Other']
    water_mask = ~df_summary['water_type_clean'].isin(exclude_categories)
    
    df_summary_before = len(df_summary)
    df_summary = df_summary[water_mask].copy()
    df_summary_after = len(df_summary)
    
    excluded_count = df_summary_before - df_summary_after
    print(f"\nFiltering results: {df_summary_before:,} → {df_summary_after:,} rows (excluded: {excluded_count:,})")
    
else:
    print("WARNING: 'water_type_clean' column not found!")

# Set up consistent color palette (same as main CA notebook)
print(f"\n=== COLOR PALETTE SETUP ===")

colorblind_safe_colors = [
    '#4477AA',  # blue
    '#EE6677',  # red
    '#228833',  # green  
    '#CCBB44',  # yellow
    '#66CCEE',  # cyan
    '#AA3377',  # purple
    '#332288',  # dark purple
    '#99DDFF',  # light blue
    '#44BB99',  # teal
    '#DDCC77'   # beige
]

def create_color_mapping(data, cat_col):
    """Create color mapping based on category frequency"""
    cat_counts = data[cat_col].value_counts()
    cats_by_freq = cat_counts.index.tolist()
    color_mapping = dict(zip(cats_by_freq, colorblind_safe_colors[:len(cats_by_freq)]))
    
    print("Color mapping (most to least common):")
    for cat in cats_by_freq:
        print(f"  {cat}: {color_mapping[cat]} ({cat_counts[cat]:,} points)")
    
    return color_mapping, cats_by_freq

# Create initial GeoDataFrame for color setup
gdf_temp = gpd.GeoDataFrame(
    df_summary,
    geometry=gpd.points_from_xy(df_summary["lon_combined"], df_summary["lat_combined"]),
    crs="EPSG:4326"
)

# Set up global color mapping
GLOBAL_COLOR_MAPPING, GLOBAL_CATS_BY_FREQ = create_color_mapping(gdf_temp, 'water_type_clean')
print("="*50)

## Filter to Santa Barbara County

In [ ]:
# ==== Filter to Santa Barbara County using shapefile ====
import geopandas as gpd
import os

print("=== SANTA BARBARA COUNTY FILTERING ===")

# Path to your Santa Barbara County shapefile
sb_county_shapefile = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\sb_county_shapefile\sb_county_shp.shp"

# Load Santa Barbara County shapefile
if not os.path.exists(sb_county_shapefile):
    raise FileNotFoundError(f"Santa Barbara County shapefile not found: {sb_county_shapefile}")

sb_county = gpd.read_file(sb_county_shapefile)
print(f"Loaded Santa Barbara County shapefile with {len(sb_county)} feature(s)")

# Ensure it's in WGS84
if sb_county.crs is None:
    raise RuntimeError("Santa Barbara County shapefile has no CRS defined. Please check the .prj file.")
if sb_county.crs.to_string().upper() not in ("EPSG:4326", "WGS84"):
    sb_county = sb_county.to_crs("EPSG:4326")

# Create unified county geometry
sb_county_geom = sb_county.union_all() if hasattr(sb_county, "union_all") else sb_county.unary_union

# Create GeoDataFrame from your data
gdf = gpd.GeoDataFrame(
    df_summary,
    geometry=gpd.points_from_xy(df_summary["lon_combined"], df_summary["lat_combined"]),
    crs="EPSG:4326"
)

# Filter points to Santa Barbara County
within_sb = gdf.within(sb_county_geom)
gdf_sb = gdf.loc[within_sb].copy()

print(f"\nFiltering results:")
print(f"  Total CA sites: {len(gdf):,}")
print(f"  Inside Santa Barbara County: {len(gdf_sb):,}")
print(f"  Outside Santa Barbara County: {(~within_sb).sum():,}")

if len(gdf_sb) == 0:
    print("\nWARNING: No sites found in Santa Barbara County!")
else:
    print(f"\nSanta Barbara County water type distribution:")
    sb_water_counts = gdf_sb['water_type_clean'].value_counts(dropna=False)
    print(sb_water_counts)

# Preview some Santa Barbara sites
preview_cols = [c for c in ["site", "site_id", "name", "station", "water_type_clean", "coord_source", "lat_combined", "lon_combined"] if c in gdf_sb.columns]
if not gdf_sb.empty:
    print("\nPreview of Santa Barbara County sites:")
    display(gdf_sb[preview_cols].head(10) if preview_cols else gdf_sb.head(10))

# Save extent for consistent mapping (with padding for better visualization)
if not gdf_sb.empty:
    minx, miny, maxx, maxy = gdf_sb.total_bounds
    pad_lon = max(0.01, (maxx - minx) * 0.15)  # County-level padding
    pad_lat = max(0.01, (maxy - miny) * 0.15)
    SB_XMIN, SB_YMIN, SB_XMAX, SB_YMAX = (minx - pad_lon, miny - pad_lat, maxx + pad_lon, maxy + pad_lat)
    print(f"\nSanta Barbara County map extent: {SB_XMIN:.3f}, {SB_YMIN:.3f}, {SB_XMAX:.3f}, {SB_YMAX:.3f}")

# Store the county geometry for mapping
SB_COUNTY_GEOM = sb_county_geom
SB_COUNTY_GDF = gpd.GeoSeries([sb_county_geom], crs="EPSG:4326")

## Output directory setup

In [ ]:
# Create output directory for Santa Barbara County maps
outdir = os.path.join("plots", run_id, "santa_barbara_county")
os.makedirs(outdir, exist_ok=True)
print(f"Output directory: {outdir}")

# Export Santa Barbara County data for reference
if not gdf_sb.empty:
    sb_csv = os.path.join(outdir, "santa_barbara_sites.csv")
    gdf_sb.drop(columns="geometry").to_csv(sb_csv, index=False)
    print(f"Exported Santa Barbara County sites to: {sb_csv}")

## Mapping 

In [ ]:
# ==== (1) Santa Barbara County - All Water Types Map ====
import os
import matplotlib.pyplot as plt
import geopandas as gpd

print("=== CREATING SANTA BARBARA COUNTY WATER TYPE MAP ===")

# Check if we have data
if 'gdf_sb' not in globals() or gdf_sb.empty:
    print("ERROR: No Santa Barbara County data found. Run the filtering cell first.")
else:
    print(f"Creating map with {len(gdf_sb):,} Santa Barbara County sites")
    
    # Create the map
    fig, ax = plt.subplots(figsize=(10, 8), dpi=300)
    
    # Add California state context in light gray
    try:
        # Load California shapefile for context
        ca_context_path = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\ca_shapefile\ca_poly.shp"
        if os.path.exists(ca_context_path):
            ca_context = gpd.read_file(ca_context_path).to_crs("EPSG:4326")
            ca_context_geom = ca_context.union_all() if hasattr(ca_context, "union_all") else ca_context.unary_union
            gpd.GeoSeries([ca_context_geom], crs="EPSG:4326").plot(
                ax=ax,
                color="#f0f0f0",  # Very light gray for context
                edgecolor="#e0e0e0",
                linewidth=0.5,
                zorder=0
            )
            print("Added California state context")
    except Exception as e:
        print(f"Could not load CA context: {e}")
    
    # Plot Santa Barbara County boundary (darker than state context)
    SB_COUNTY_GDF.plot(
        ax=ax, 
        color="lightgray", 
        edgecolor="black", 
        linewidth=0.8, 
        zorder=1
    )
    
    # Plot points by frequency using consistent colors (most common first, on bottom)
    for cat in GLOBAL_CATS_BY_FREQ:
        sub = gdf_sb[gdf_sb['water_type_clean'] == cat]
        if len(sub) == 0:
            continue
        sub.plot(
            ax=ax,
            color=GLOBAL_COLOR_MAPPING[cat],
            markersize=15,  # Larger markers as requested
            alpha=0.8,
            edgecolor='none',  # Remove black border
            linewidth=0,
            zorder=2,
            label=cat
        )
    
    # Set extent
    ax.set_xlim(SB_XMIN, SB_XMAX)
    ax.set_ylim(SB_YMIN, SB_YMAX)
    
    # Clean styling
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.grid(False)
    
    # Add border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#cccccc')
        spine.set_linewidth(1.0)
    
    # Legend (no title on figure)
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        ncol=1,
        markerscale=1.5,
        frameon=True,
        fancybox=True,
        framealpha=0.9
    )
    
    # Save
    outfile = os.path.join(outdir, "SB_county_all_water_types.png")
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    print(f"Saved: {outfile}")
    plt.show()

# ==== (2) Santa Barbara County - Decade Panel Map ====
print("\n=== CREATING SANTA BARBARA COUNTY DECADE MAPS ===")

if 'gdf_sb' not in globals() or gdf_sb.empty:
    print("ERROR: No Santa Barbara County data found.")
elif 'decade' not in gdf_sb.columns:
    print("ERROR: No 'decade' column found in data.")
else:
    # Filter to available decades
    available_decades = sorted(gdf_sb["decade"].dropna().unique())
    print(f"Found {len(available_decades)} decades in Santa Barbara County: {available_decades}")
    
    if len(available_decades) == 0:
        print("No decade data available for Santa Barbara County")
    else:
        # Determine layout based on number of decades
        n_decades = len(available_decades)
        if n_decades <= 6:
            ncols, nrows = 3, 2
        elif n_decades <= 9:
            ncols, nrows = 3, 3
        elif n_decades <= 12:
            ncols, nrows = 4, 3
        else:
            ncols, nrows = 4, 4
        
        total_panels = ncols * nrows
        
        print(f"Creating {ncols}x{nrows} layout for {n_decades} decades")
        
        # Create figure
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, 8), dpi=300)
        axes = axes.flatten() if isinstance(axes, list) or hasattr(axes, 'flatten') else [axes]
        
        # Adjust spacing
        plt.subplots_adjust(
            left=0.05,
            bottom=0.05,
            right=0.85,  # Leave space for legend
            top=0.95,
            wspace=0.1,
            hspace=0.15
        )
        
        # Plot each decade
        for idx, decade in enumerate(available_decades):
            if idx >= len(axes):
                break
                
            ax = axes[idx]
            sub = gdf_sb[gdf_sb["decade"] == decade]
            
            # Add California state context in light gray
            try:
                ca_context_path = r"C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\data\ca_shapefile\ca_poly.shp"
                if os.path.exists(ca_context_path):
                    ca_context = gpd.read_file(ca_context_path).to_crs("EPSG:4326")
                    ca_context_geom = ca_context.union_all() if hasattr(ca_context, "union_all") else ca_context.unary_union
                    gpd.GeoSeries([ca_context_geom], crs="EPSG:4326").plot(
                        ax=ax,
                        color="#f0f0f0",  # Very light gray for context
                        edgecolor="#e0e0e0",
                        linewidth=0.3,  # Thinner lines for small panels
                        zorder=0
                    )
            except Exception:
                pass  # Skip CA context if can't load
            
            # County background (darker than state context)
            SB_COUNTY_GDF.plot(
                ax=ax, 
                color="lightgray", 
                edgecolor="black", 
                linewidth=0.5, 
                zorder=1
            )
            
            # Plot points by frequency using consistent colors
            for cat in GLOBAL_CATS_BY_FREQ:
                s = sub[sub['water_type_clean'] == cat]
                if len(s) == 0:
                    continue
                s.plot(
                    ax=ax,
                    color=GLOBAL_COLOR_MAPPING[cat],
                    markersize=10,
                    alpha=0.8,
                    edgecolor='none',
                    linewidth=0,
                    zorder=2
                )
            
            # Set extent
            ax.set_xlim(SB_XMIN, SB_XMAX)
            ax.set_ylim(SB_YMIN, SB_YMAX)
            ax.set_aspect('equal')
            
            # Title with decade and count (no overall title)
            count = len(sub)
            ax.set_title(f"{int(decade)}s ({count} sites)", fontsize=10, pad=5)
            
            # Clean styling
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel("")
            ax.set_ylabel("")
            ax.grid(False)
            
            # Light border
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_color('#dddddd')
                spine.set_linewidth(0.5)
        
        # Hide unused panels
        for idx in range(len(available_decades), len(axes)):
            axes[idx].axis('off')
        
        # Add shared legend
        from matplotlib.lines import Line2D
        handles = [Line2D([0],[0], marker='o', linestyle='',
                          markerfacecolor=GLOBAL_COLOR_MAPPING[c], 
                          markeredgecolor='none',
                          markersize=8, label=c)
                   for c in GLOBAL_CATS_BY_FREQ if c in gdf_sb['water_type_clean'].values]
        
        fig.legend(
            handles=handles,
            loc='center left',
            bbox_to_anchor=(0.87, 0.5),
            frameon=True,
            fancybox=True,
            framealpha=0.9,
            fontsize=9
        )
        
        # Overall title removed - no suptitle
        
        # Save
        outfile = os.path.join(outdir, "SB_county_by_decade.png")
        plt.savefig(outfile, dpi=300, bbox_inches="tight")
        print(f"Saved: {outfile}")
        plt.show()